# Lab 3 — FIR filters and analysis–synthesis filter banks

**Audio Coding · Topic 3 · Student lab**

This lab follows the concepts in the lecture and complements the examples `FIR-design`, `filter-bank-basic`, and `filter-bank-critically-sampled`. You will design a filter, assemble a bank of frequency-selective filters, and investigate what changes when the subbands are **critically sampled**.

**Learning goals**

1. Relate FIR length, cutoff frequency, frequency response and audible filtering.
2. Explain how modulation of a low-pass prototype creates uniform frequency subbands.
3. Identify explicitly the **analysis → ↓M → ↑M → synthesis** stages.
4. Check the coefficient count and distinguish **critical sampling** from **perfect reconstruction**.

**Student work:** Code marked `TODO` is for you to complete. Questions have no answers here. You should submit your modified notebook with plots, numerical results and concise explanations. This is not an implementation of a production PQMF or a perfect-reconstruction bank.

**Audio files:** The examples use `../sounds/white-noise.wav` and `../sounds/487465-trumpet.wav` when available. If not found, reproducible demonstration signals will be generated so that the lab can still run.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import signal
from scipy.io import wavfile
from pathlib import Path
from IPython.display import Audio, display

EPS = 1e-12

def read_sound(filename, kind="harmonic", fs=16000, duration=2.0):
    """Load WAV as mono float, or use a clearly identified reproducible fallback."""
    path = Path(filename)
    if path.exists():
        fs, x = wavfile.read(path)
        if x.ndim > 1:
            x = x.astype(np.float64).mean(axis=1)
        if np.issubdtype(x.dtype, np.integer):
            info = np.iinfo(x.dtype)
            if info.min == 0:  # unsigned PCM (typically 8-bit)
                x = (x.astype(float) - (info.max + 1)/2)/((info.max + 1)/2)
            else:
                x = x.astype(float)/max(abs(info.min), abs(info.max))
        else:
            x = x.astype(float)
        print(f"Loaded {path} ({fs} Hz)")
        return x, fs
    print(f"Missing {path}: using a synthetic {kind} signal instead.")
    t = np.arange(round(fs*duration))/fs
    if kind == "noise":
        x = np.random.default_rng(7).standard_normal(t.size)
        x = 0.3*x/np.max(np.abs(x))
    else:
        x = (0.5*np.sin(2*np.pi*220*t) + 0.25*np.sin(2*np.pi*440*t)
             + 0.12*np.sin(2*np.pi*880*t)) * (0.6+0.4*np.sin(np.pi*t/duration)**2)
    return x, fs

def response(h, fs, title=None):
    f, H = signal.freqz(h, worN=4096, fs=fs)
    plt.figure(figsize=(9, 3))
    plt.plot(f, 20*np.log10(np.maximum(np.abs(H), EPS)))
    plt.ylim(-100, 10); plt.xlabel('Frequency (Hz)'); plt.ylabel('Gain (dB)')
    plt.grid(True)
    if title: plt.title(title)
    plt.tight_layout(); plt.show()
    return f, H

def listen_pair(x, y, fs):
    """Apply one common playback gain; never normalize the two signals independently."""
    peak = max(np.max(np.abs(x)), np.max(np.abs(y)), EPS)
    scale = min(1.0, 0.9/peak)
    print('Original:'); display(Audio(scale*x, rate=int(fs), normalize=False))
    print('Processed:'); display(Audio(scale*y, rate=int(fs), normalize=False))

def prototype_filter(N, M):
    """Simple windowed-sinc prototype; illustrative, not a designed PQMF prototype."""
    return signal.firwin(N, 1/(2*M), window='hamming')

def modulate_filter(h, M, k):
    """Modulate an FIR prototype around channel center (k+0.5)*fs/(2*M)."""
    n = np.arange(len(h))
    return 2*h*np.cos(np.pi*(k+0.5)*(n-(len(h)-1)/2)/M)

def filters_for_bank(M, N):
    h = prototype_filter(N, M)
    return np.array([modulate_filter(h, M, k) for k in range(M)])

def analysis_bank(x, filters, M):
    """Full convolution retains filter tails; decimate each channel by M."""
    return [signal.fftconvolve(x, hk, mode='full')[::M] for hk in filters]

def synthesis_bank(subbands, filters, M):
    """Zero insert, filter each branch, sum. No promise of perfect reconstruction."""
    max_len = max(M*(len(c)-1)+1+len(hk)-1 for c, hk in zip(subbands, filters))
    y = np.zeros(max_len)
    for c, hk in zip(subbands, filters):
        up = np.zeros(M*(len(c)-1)+1)
        up[::M] = c
        branch = signal.fftconvolve(up, hk, mode='full')
        y[:len(branch)] += branch
    return y

def snr_db(x, y):
    error = x-y
    return 10*np.log10((np.sum(x*x)+EPS)/(np.sum(error*error)+EPS))


## Part 1 — FIR design and listening

Review the lecture's **FIR filters** and **window-method (`firwin`)** slides, and the `FIR-design` example. Filter order and filter length differ by one: an FIR with `N` taps has order `N - 1`.

Design a low-pass filter, plot its impulse and frequency responses, and listen to its effect on noise. Frequencies in `firwin(..., fs=fs)` are specified in Hz.

In [ ]:
noise, fs = read_sound('../sounds/white-noise.wav', kind='noise')
N = 201
cutoff_hz = 2000
h = signal.firwin(N, cutoff_hz, fs=fs, window='hamming')
plt.figure(figsize=(9, 2.5)); plt.plot(h); plt.title('FIR impulse response')
plt.xlabel('Tap index'); plt.ylabel('Amplitude'); plt.tight_layout(); plt.show()
response(h, fs, f'FIR low-pass ({N} taps, cutoff {cutoff_hz} Hz)')
filtered = signal.lfilter(h, [1], noise)
listen_pair(noise, filtered, fs)
print('The causal linear-phase FIR group delay is', (N-1)/2, 'samples.')

**Task 1 — Filter length and selectivity**

1. Repeat the 2 kHz low-pass design with **31, 101 and 401** taps, keeping the same cutoff, window and sample rate. Overlay their frequency responses. What happens to transition width, sidelobes and delay?
2. Compare the original noise with two filtered versions by listening. Describe the audible difference.
3. Explain why increasing the filter length does not *necessarily* improve the nominal stopband sidelobe level when the window type stays the same.

Use the cell below for your results.

In [ ]:
# TODO — Task 1: try different lengths and overlay their responses.
lengths = [31, 101, 401]
# for taps in lengths:
#     ...

## Part 2 — From one filter to a uniform filter bank

The lecture's **uniform filter bank** and **cosine-modulated filter bank** slides motivate this section. We use one low-pass prototype and cosine modulation to obtain `M` passbands with approximate center frequencies

$$
f_k = \left(k+\frac{1}{2}\right)\frac{f_s}{2M},\qquad k=0,\ldots,M-1.
$$

Here, `M` is the **number of frequency bands**, whereas `N` is the **prototype length in taps**. These are distinct parameters. The following simple modulation does **not** implement the more carefully phase-adjusted PQMF filter bank from the lecture.

**Important:** In this part all filtered outputs remain at the original sampling rate. Simply adding them is **not** an analysis–synthesis bank and does not guarantee reconstruction.

In [ ]:
x, fs = read_sound('../sounds/487465-trumpet.wav')
M, N = 8, 257
filters = filters_for_bank(M, N)
plt.figure(figsize=(10, 4))
for k, hk in enumerate(filters):
    f, Hk = signal.freqz(hk, worN=4096, fs=fs)
    plt.plot(f, 20*np.log10(np.maximum(np.abs(Hk), EPS)), label=f'{k}')
plt.xlabel('Frequency (Hz)'); plt.ylabel('Gain (dB)'); plt.ylim(-85, 10)
plt.title('Cosine-modulated channels'); plt.grid(True); plt.tight_layout(); plt.show()
k = 2
one_band = signal.lfilter(filters[k], [1], x)
print(f'Band {k}: nominal center {(k+.5)*fs/(2*M):.1f} Hz')
listen_pair(x, one_band, fs)

**Task 2 — Subband analysis**

1. Select two well-separated channels and compare their frequency responses and sounds. Which frequency components are retained?
2. Repeat the design for `M = 16` with a suitably longer prototype (`N = 513` is a starting point). Explain why changing `M` also changes the ideal passband spacing.
3. Compute and plot both `abs(sum(H_k))` and `sum(abs(H_k))` for the full-rate filter bank. Explain why they are different, and which one corresponds to the frequency response of a **sum of filtered signals**.
4. Does summing the full-rate subbands reconstruct the original exactly? Support the answer with a plot or sound comparison.

Complete the cell below without replacing the complex frequency response by its magnitude before summation.

In [ ]:
# TODO — Task 2: complex response summation and full-rate audio sum.
# responses = []
# for hk in filters:
#     f, Hk = signal.freqz(hk, worN=4096, fs=fs)
#     responses.append(Hk)
# ...

## Part 3 — Critical sampling: analysis → ↓M → ↑M → synthesis

This part connects directly to the lecture's **downsampling**, **upsampling** and **uniform maximally decimated filter bank** diagrams, and to the `filter-bank-critically-sampled` example.

For each channel:

1. **Analysis:** filter input at the original rate.
2. **Downsample by `M`:** keep every `M`-th filtered sample (`filtered[::M]`).
3. **Upsample by `M`:** insert `M-1` zeros between successive stored values.
4. **Synthesis:** filter the expanded sequence, then sum the channel outputs.

The total coefficient rate is approximately

$$
M\frac{f_s}{M}=f_s.
$$

Thus the bank is **critically sampled**: in steady state, its aggregate number of subband coefficients per second equals the input sample rate. Padding and filter tails can add a small number of boundary coefficients.

**Critical sampling does not imply perfect reconstruction.** Correct alias cancellation requires appropriately designed analysis and synthesis filters. Our basic cosine-modulated filters do not guarantee that property.

In [ ]:
# Visualize exactly what happens in one branch.
M, N, k = 8, 257, 2
filters = filters_for_bank(M, N)
full_band = signal.fftconvolve(x, filters[k], mode='full')
lowrate = full_band[::M]                 # ↓M: decimation
zero_inserted = np.zeros(M*(len(lowrate)-1)+1)
zero_inserted[::M] = lowrate            # ↑M: zero insertion
synthesized_branch = signal.fftconvolve(zero_inserted, filters[k], mode='full')
print('One filtered branch:', len(full_band), 'samples')
print('Stored after decimation:', len(lowrate), 'samples')
print('Upsampled sequence:', len(zero_inserted), 'samples')
fig, axes = plt.subplots(3, 1, figsize=(10, 6))
axes[0].plot(full_band[:160]); axes[0].set_title('Analysis filtered (original sample grid)')
axes[1].stem(np.arange(20), lowrate[:20], basefmt=' '); axes[1].set_title('Downsampled (one sample in M)')
axes[2].stem(np.arange(80), zero_inserted[:80], basefmt=' '); axes[2].set_title('Upsampled (zero insertion)')
for a in axes: a.set_xlabel('Sample index in the corresponding sequence')
plt.tight_layout(); plt.show()

In [ ]:
# Check complete analysis/synthesis and align the nominal FIR delay.
subbands = analysis_bank(x, filters, M)
y_full = synthesis_bank(subbands, filters, M)
# For symmetric N-tap FIR filters, analysis plus synthesis has nominal delay N-1.
delay = N-1
y_aligned = y_full[delay:delay+len(x)]
assert len(y_aligned) == len(x)
print('Input samples:', len(x))
print('Stored coefficients:', sum(map(len, subbands)))
print('Stored/input ratio:', sum(map(len, subbands))/len(x))
print('Nominal delay:', delay, 'samples')
print('SNR after nominal alignment:', f'{snr_db(x, y_aligned):.2f} dB')
listen_pair(x, y_aligned, fs)
plt.figure(figsize=(10, 3))
plt.plot(np.arange(min(1000,len(x)))/fs, x[:1000], label='Input', alpha=.8)
plt.plot(np.arange(min(1000,len(x)))/fs, y_aligned[:1000], label='Reconstructed', alpha=.8)
plt.xlabel('Time (s)'); plt.legend(); plt.tight_layout(); plt.show()

**Task 3 — Critical sampling and reconstruction**

1. Show numerically that the total number of saved subband coefficients is approximately the number of input samples. Explain any boundary difference.
2. Explain why the sampling rate of **one** decimated subband is `fs/M`, but playing one subband as if it were full-band audio would be misleading.
3. Repeat the complete analysis/synthesis experiment with `(M, N) = (8, 257)` and `(16, 513)`. Record the coefficient counts, output SNR and listening observations. Which aspects change?
4. Use the code to locate the downsampling and zero-insertion operations. Explain their spectral consequences: **aliasing** during decimation and **images** during upsampling.
5. Is a low reconstruction SNR evidence that critical sampling is impossible? Explain the difference between this illustrative bank and a properly designed near-perfect-reconstruction PQMF bank.

**Optional extension:** Inspect the phase-adjusted analysis/synthesis modulation equations in the lecture. Explain why simply using the same real-valued filter for both stages need not cancel aliasing.

In [ ]:
# TODO — Task 3: compare (M, N) designs.
settings = [(8, 257), (16, 513)]
# for M_test, N_test in settings:
#     ...

## What to remember

- FIR length, window and cutoff control selectivity and delay; design using the **actual sample rate**.
- Cosine modulation turns a low-pass prototype into a set of subbands, but does not automatically yield a reconstructing bank.
- **Downsampling** stores fewer samples per band; **upsampling** inserts zeros and requires synthesis filtering.
- A bank with `M` channels and decimation by `M` is **critically sampled** in steady state.
- **Critical sampling and perfect reconstruction are different properties**. Filter design and alias cancellation determine reconstruction quality.

**Related course material:** *Topic 3 — Time-frequency representations: filter banks and transforms*; `FIR-design.ipynb`, `filter-bank-basic.ipynb`, `filter-bank-critically-sampled.ipynb`.